# Milestone 1: Data Profiling

**Tegar Reski Pratama — 202310370311215**

Adaptasi template UMM-GURU untuk *Indonesian News Articles Published at 2017*.
Unduh `mq_news.csv` sesuai `data/README.md`, lalu jalankan setiap cell dari atas.
File ini disiapkan tanpa execution count dan tanpa output.


In [1]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents)
     if (path / 'requirements.txt').is_file() and (path / 'data/README.md').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Buka Jupyter dari akar repository atau folder notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
from IPython.display import Markdown, display
from src.common import (
    connect_analytics, prepare_directories, read_json, sha256_file,
    sink_parquet_atomic, sql_literal, write_json,
)
prepare_directories(PROJECT_ROOT)

from src.dataset import DATASET_URL, missing_summary, scan_news_csv

DATA_CANDIDATES = [
    PROJECT_ROOT / 'data/raw/Dataset Indonesian News Articles Published at 2017/mq_news.csv',
    PROJECT_ROOT / 'data/raw/mq_news.csv',
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), DATA_CANDIDATES[0])
RAW_PARQUET = PROJECT_ROOT / 'data/processed/dataset_raw.parquet'
# Pemetaan otomatis mengenali date/title/content/source dan variasi headernya.
# Dictionary kosong memang berarti memakai pemetaan otomatis.
COLUMN_OVERRIDES = {}

if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Unduh CSV asli dari {DATASET_URL} ke {DATA_PATH}.')

raw_csv, lazy_data, column_mapping = scan_news_csv(DATA_PATH, COLUMN_OVERRIDES)
print('Header CSV:', raw_csv.collect_schema().names())
print('Pemetaan kolom:', column_mapping)
print(f'Ukuran CSV: {DATA_PATH.stat().st_size / 1_000_000:,.2f} MB')


Header CSV: ['', 'item_id', 'url', 'title', 'content_html', 'published_at', 'source']
Pemetaan kolom: {'published_raw': 'published_at', 'title_raw': 'title', 'content_raw': 'content_html', 'source_raw': 'source', 'url_raw': 'url'}
Ukuran CSV: 1,371.44 MB


## Profil Polars

`scan_csv` memakai lazy evaluation. Kolom CSV dibaca sebagai String terlebih dahulu
agar inferensi tipe tidak merusak teks atau tanggal; konversi tipe dilakukan saat cleaning.


In [2]:
raw_schema = raw_csv.collect_schema()
row_count = int(lazy_data.select(pl.len()).collect(engine='streaming').item())
size_bytes = DATA_PATH.stat().st_size
size_requirement_met = size_bytes >= 500_000_000 or row_count > 1_000_000
ringkasan = pl.DataFrame({
    'jumlah_baris': [row_count], 'jumlah_kolom_csv': [len(raw_schema)],
    'ukuran_csv_MB': [size_bytes / 1_000_000],
    'memenuhi_syarat_ukuran': [size_requirement_met],
})
display(ringkasan)
assert row_count > 0, 'CSV tidak memiliki artikel.'
assert size_requirement_met, 'CSV harus >= 500 MB atau memiliki > 1.000.000 baris.'


jumlah_baris,jumlah_kolom_csv,ukuran_csv_MB,memenuhi_syarat_ukuran
i64,i64,f64,bool
455678,7,1371.437745,true


In [3]:
null_summary = missing_summary(
    lazy_data, ['published_raw', 'title_raw', 'content_raw', 'source_raw', 'url_raw']
)
blank_summary = lazy_data.select([
    pl.col(name).str.strip_chars().eq('').fill_null(False).sum().alias(name)
    for name in ['published_raw', 'title_raw', 'content_raw', 'source_raw', 'url_raw']
]).collect(engine='streaming')
display(null_summary)
display(blank_summary)
null_summary.write_csv(PROJECT_ROOT / 'output/nulls_raw.csv')


kolom,jumlah_null,persen_null
str,i64,f64
"""published_raw""",0,0.0
"""title_raw""",0,0.0
"""content_raw""",0,0.0
"""source_raw""",0,0.0
"""url_raw""",0,0.0


published_raw,title_raw,content_raw,source_raw,url_raw
u32,u32,u32,u32,u32
0,0,1,0,0


In [4]:
contoh_data = lazy_data.head(10).with_columns(
    pl.col('content_raw').str.slice(0, 160).alias('content_preview')
).select('row_id', 'published_raw', 'title_raw', 'source_raw', 'content_preview')
contoh_data.collect(engine='streaming')


row_id,published_raw,title_raw,source_raw,content_preview
u32,str,str,str,str
0,"""2017-01-01 00:00:00""","""Tips Agar Resolusi Tahun Baru …","""liputan6.com""","""<div class=""article-content-bo…"
1,"""2017-01-01 00:01:00""","""Tragedi Mendebarkan di Penghuj…","""liputan6.com""","""<div class=""article-content-bo…"
2,"""2017-01-01 00:03:09""","""Pantau Tahun Baru, Kapolda Met…","""kompas.com""","""<p><strong>JAKARTA, KOMPAS.com…"
3,"""2017-01-01 00:04:21""","""Terompet dan Kembang Api Bersa…","""liputan6.com""","""<div class=""article-content-bo…"
4,"""2017-01-01 00:05:08""","""Awas! Copet Berkeliaran di Anc…","""liputan6.com""","""<div class=""article-content-bo…"
5,"""2017-01-01 00:05:54""","""Dua Gol Kejutan Manchester Uni…","""tribunnews.com""","""<p><strong>TRIBUNNEWS.COM - </…"
6,"""2017-01-01 00:16:52""","""Diego Costa Cetak Gol Pamungka…","""tribunnews.com""","""<p><strong>TRIBUNNEWS.COM - </…"
7,"""2017-01-01 00:16:54""","""Rayakan Malam Pergantian Tahun…","""kompas.com""","""<p><strong>AMBON,KOMPAS.com - …"
8,"""2017-01-01 00:20:52""","""Plt Gubernur DKI Tembakkan Sua…","""detik.com""","""<b>Jakarta</b> -Plt Gubernur D…"


## Konversi CSV ke Parquet

Simpan salinan kolom analisis beserta `row_id` dengan sink streaming.
CSV asli dipertahankan; checksum mengidentifikasi file input secara persis.


In [5]:
raw_csv_sha256 = sha256_file(DATA_PATH)
sink_parquet_atomic(lazy_data, RAW_PARQUET)
raw_parquet_sha256 = sha256_file(RAW_PARQUET)
print(f'Parquet tersimpan: {RAW_PARQUET.relative_to(PROJECT_ROOT)}')
print(f'SHA-256 CSV: {raw_csv_sha256}')


Parquet tersimpan: data/processed/dataset_raw.parquet
SHA-256 CSV: 32f0839ab07ada289c23bfff3508c8753037ef6d345e848c3c59558db531a953


## Profil DuckDB

`SUMMARIZE` menghasilkan statistik tipe, null, min/max, dan perkiraan nilai unik.
Nilai unik pada output ini bersifat aproksimasi; deduplikasi memakai kunci artikel secara eksplisit.


In [6]:
connection = connect_analytics(PROJECT_ROOT)
connection.execute(
    f'CREATE VIEW raw_news AS SELECT * FROM read_parquet({sql_literal(RAW_PARQUET)})'
)
profil_duckdb = connection.execute('SUMMARIZE SELECT * FROM raw_news').pl()
profil_duckdb.write_csv(PROJECT_ROOT / 'output/profile_raw.csv')
display(profil_duckdb)
assert connection.execute('SELECT COUNT(*) FROM raw_news').fetchone()[0] == row_count
connection.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""row_id""","""UINTEGER""","""0""","""455677""",475106,"""227838.5""","""131543.05231938255""","""115522""","""229570""","""340747""",455678,0.00
"""published_raw""","""VARCHAR""","""2017-01-01 00:00:00""","""2017-12-31 23:58:25""",401317,null,null,null,null,null,455678,0.00
"""title_raw""","""VARCHAR""",""""" Di Sini Punya Prinsip, Jika …","""﻿﻿Waspada akan Diabetes Mellit…",526630,null,null,null,null,null,455678,0.00
"""content_raw""","""VARCHAR""","""&amp;ldquo;Siapa di antara kit…",""" """,419110,null,null,null,null,null,455678,0.00
"""source_raw""","""VARCHAR""","""beritasatu.com""","""tribunnews.com""",6,null,null,null,null,null,455678,0.00
"""url_raw""","""VARCHAR""","""http://edukasi.kompas.com/read…","""https://www.tempo.co/read/1046…",390541,null,null,null,null,null,455678,0.00


## Catatan Milestone 1

Ringkasan berikut dihitung dari CSV lokal dan disimpan untuk pemeriksaan notebook 02.
Kolom tambahan di luar tanggal, judul, isi, sumber, dan URL tidak disalin ke Parquet analisis;
header asli tetap tercatat dalam metadata dan CSV asli.


In [7]:
profiling_summary = {
    'dataset_url': DATASET_URL,
    'raw_csv': DATA_PATH.relative_to(PROJECT_ROOT).as_posix(), 'raw_size_bytes': size_bytes,
    'rows_raw': row_count, 'csv_columns': dict((k, str(v)) for k, v in raw_schema.items()),
    'column_mapping': column_mapping, 'size_requirement_met': size_requirement_met,
    'raw_csv_sha256': raw_csv_sha256, 'raw_parquet_sha256': raw_parquet_sha256,
    'null_counts': null_summary.to_dicts(),
    'blank_counts': blank_summary.to_dicts()[0],
    'polars_version': pl.__version__,
}
write_json(PROJECT_ROOT / 'output/profiling_summary.json', profiling_summary)
highest_null = null_summary.row(0, named=True)
display(Markdown(
    f"CSV berisi **{row_count:,} artikel**, berukuran **{size_bytes / 1_000_000:,.2f} MB**. "
    f"Syarat skala: **{size_requirement_met}**. Kolom kanonis dengan null terbanyak: "
    f"**{highest_null['kolom']}** ({highest_null['jumlah_null']:,} null). "
    "Kolom opsional yang tidak tersedia akan tampil sebagai null; notebook 02 "
    "memeriksa tanggal, teks kosong, duplikat, dan panjang artikel."
))


CSV berisi **455,678 artikel**, berukuran **1,371.44 MB**. Syarat skala: **True**. Kolom kanonis dengan null terbanyak: **published_raw** (0 null). Kolom opsional yang tidak tersedia akan tampil sebagai null; notebook 02 memeriksa tanggal, teks kosong, duplikat, dan panjang artikel.